<h1> Homework 3
<h3> Mirko Dugajlić


<h2> Dataset import and preprocessing

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV
#from tqdm.notebook import trange, tqdm
#import statsmodels.api as sm
#import statsmodels.api
import seaborn as sns
#from statsmodels.tools.tools import maybe_unwrap_results
#from statsmodels.graphics.gofplots import ProbPlot
#from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
from typing import Type
from sklearn.utils import resample

In [2]:
column_names = [
    "state", "county", "community", "communityname", "fold", "population", "householdsize", "racepctblack", "racePctWhite", "racePctAsian", "racePctHisp", "agePct12t21", "agePct12t29", "agePct16t24", "agePct65up", "numbUrban", "pctUrban", "medIncome", "pctWWage", "pctWFarmSelf",
    "pctWInvInc", "pctWSocSec", "pctWPubAsst", "pctWRetire", "medFamInc", "perCapInc", "whitePerCap", "blackPerCap", "indianPerCap", "AsianPerCap", "OtherPerCap", "HispPerCap", "NumUnderPov",  "PctPopUnderPov", "PctLess9thGrade", "PctNotHSGrad", "PctBSorMore", "PctUnemployed", 
    "PctEmploy", "PctEmplManu", "PctEmplProfServ", "PctOccupManu", "PctOccupMgmtProf",  "MalePctDivorce", "MalePctNevMarr", "FemalePctDiv", "TotalPctDiv", "PersPerFam", "PctFam2Par", "PctKids2Par", "PctYoungKids2Par", "PctTeen2Par", "PctWorkMomYoungKids", "PctWorkMom", "NumIlleg", 
    "PctIlleg", "NumImmig", "PctImmigRecent", "PctImmigRec5", "PctImmigRec8", "PctImmigRec10", "PctRecentImmig", "PctRecImmig5", "PctRecImmig8", "PctRecImmig10", "PctSpeakEnglOnly", "PctNotSpeakEnglWell", "PctLargHouseFam", "PctLargHouseOccup", "PersPerOccupHous", "PersPerOwnOccHous", 
    "PersPerRentOccHous", "PctPersOwnOccup", "PctPersDenseHous", "PctHousLess3BR", "MedNumBR", "HousVacant", "PctHousOccup", "PctHousOwnOcc", "PctVacantBoarded", "PctVacMore6Mos", "MedYrHousBuilt", "PctHousNoPhone", "PctWOFullPlumb", "OwnOccLowQuart", "OwnOccMedVal", 
    "OwnOccHiQuart", "RentLowQ", "RentMedian", "RentHighQ", "MedRent", "MedRentPctHousInc", "MedOwnCostPctInc", "MedOwnCostPctIncNoMtg", "NumInShelters", "NumStreet", "PctForeignBorn", "PctBornSameState", "PctSameHouse85", "PctSameCity85", "PctSameState85", "LemasSwornFT", 
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop", "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", "PctPolicHisp", "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits", "NumKindsDrugsSeiz", 
    "PolicAveOTWorked", "LandArea", "PopDens", "PctUsePubTrans", "PolicCars", "PolicOperBudg", "LemasPctPolicOnPatr", "LemasGangUnitDeploy", "LemasPctOfficDrugUn", "PolicBudgPerPop", "ViolentCrimesPerPop"
]



df = pd.read_csv("datasets/communities.data", names=column_names)
df = df.iloc[:, 5:]
df.replace('?', np.nan, inplace=True)
df_cleaned = df.dropna(axis=1, how='any')


X = df_cleaned

y = df_cleaned["ViolentCrimesPerPop"]
X = X.drop("ViolentCrimesPerPop", axis=1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

X.head()

,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,agePct16t24,agePct65up,...,NumStreet,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn
0,0.19,0.33,0.02,0.90,0.12,0.17,0.34,0.47,0.29,0.32,...,0.0,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20,0.32
1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,...,0.0,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45,0.00
2,0.00,0.42,0.49,0.56,0.17,0.04,0.39,0.47,0.28,0.32,...,0.0,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02,0.00
3,0.04,0.77,1.00,0.08,0.12,0.10,0.51,0.50,0.34,0.21,...,0.0,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28,0.00
4,0.01,0.55,0.02,0.95,0.09,0.05,0.38,0.38,0.23,0.36,...,0.0,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02,0.00


<h2> Ridge and Lasso regressions


In [3]:
alpha_range = {'alpha': [0.0001, 0.001, 0.003, 0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 10]}


ridge_grid = GridSearchCV(Ridge(), param_grid=alpha_range, cv=5, scoring='neg_mean_squared_error')
ridge_grid.fit(X_train, y_train)
best_ridge_model = ridge_grid.best_estimator_

lasso_grid = GridSearchCV(Lasso(), param_grid=alpha_range, cv=5, scoring='neg_mean_squared_error')
lasso_grid.fit(X_train, y_train)
best_lasso_model = lasso_grid.best_estimator_

ridge_predictions = best_ridge_model.predict(X_test)
lasso_predictions = best_lasso_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_mse = mean_squared_error(y_test, ridge_predictions)
lasso_mae = mean_absolute_error(y_test, lasso_predictions)
lasso_mse = mean_squared_error(y_test, lasso_predictions)

print("Optimized regression results:\n")
print(f"Ridge -> Best alpha: {best_ridge_model.alpha}, MAE = {ridge_mae}, MSE = {ridge_mse}")
print(f"Lasso -> Best alpha: {best_lasso_model.alpha}, MAE = {lasso_mae}, MSE = {lasso_mse}")




Optimized regression results:

Ridge -> Best alpha: 1, MAE = 0.09403409990364854, MSE = 0.017174918860066377
Lasso -> Best alpha: 0.0001, MAE = 0.09367897408399084, MSE = 0.017140222359752696


The best regularization parameters are found by searching through an array of values ranging from 0.0001 to 1,000. The model with the highest score (metric or result) will conduct the final evaluation for the comparison of these two regressions. All computations and selections were made with the help of the sklearn package. GridSearchCV is the primary regularization method, which uses cross-validation to evaluate each parameter input (from an array).


In [4]:
# Methods from the last assignment

def shuffle_data(X, y):
    indices = np.random.permutation(len(X))
    return X.iloc[indices], y.iloc[indices]

def cross_validation(X, y, n):
    X = np.array(X)
    y = np.array(y)
    
    # Split data into folds
    folds_X = np.array_split(X, n)
    folds_y = np.array_split(y, n)
    mse_scores = []

    for i in range(n):
       
        # Prepare train and test sets
        X_test, y_test = folds_X[i], folds_y[i]
        X_train = np.concatenate([folds_X[j] for j in range(n) if j != i])
        y_train = np.concatenate([folds_y[j] for j in range(n) if j != i])
        
        # Train and evaluate model
        model = LinearRegression()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        mse_scores.append(mean_squared_error(y_test, predictions))
        
    return np.mean(mse_scores)


def forward_attribute_selection(X, y, cv=5):
    best_attributes = pd.DataFrame() 
    best_mse = float('inf')
    total_mse = 0
    last_best = float('inf')
    for i in range(len(X.columns)): 
        current_best_mse = float('inf')
        current_best_candidate = None

        for column in X.columns:
            if column not in best_attributes.columns:

                # Test the candidate attribute set by adding the new column to the existing set of columns (already choosen)
                X_candidate = X[best_attributes.columns.tolist() + [column]]
                X_shuffled, y_shuffled = shuffle_data(X_candidate, y)
                mse_candidate = cross_validation(X_shuffled, y_shuffled, cv)

                # Update the best candidate if current is better
                if mse_candidate < current_best_mse:
                    current_best_mse = mse_candidate
                    current_best_candidate = X_candidate

        # Update best attributes and avg mse for "threshold"
        best_attributes = current_best_candidate
        best_mse = current_best_mse
        total_mse += best_mse
        avg_mse = total_mse / (i + 1)  # Moving average MSE

        # print(f"Iteration {i+1}: MSE={best_mse}, Avg MSE={avg_mse}")
        # Stop if the improvement falls below the threshold (average of iterations before), as discussed with the assistant on tutorials.
        # Could also break at the first occurance when current iteration-best is worse than last iter. best.
        # EDIT: I changed to the last_best threshold because of performance issues.
        # It takes more time to calculate when the average is the threshold
        if best_mse > avg_mse: #  last_best
            break
        last_best = best_mse

    print("Selected best attributes:", best_attributes.columns.tolist())
    return best_attributes


In [5]:
# Get attributes
selected_attributes = forward_attribute_selection(X_train, y_train)
print(best_lasso_model.feature_names_in_)

print(f"Number of attributes from Forward attribute selection: {len(selected_attributes.columns)}")
print(f"Number of attributes from Forward attribute selection: {len(best_lasso_model.feature_names_in_)}")

# Intersection
intersection = list(set(selected_attributes.columns) & set(best_lasso_model.feature_names_in_))
print(f"Intersection length = {len(intersection)}")

Selected best attributes: ['PctIlleg', 'FemalePctDiv', 'racePctWhite', 'HousVacant', 'PctWorkMom', 'PctKids2Par', 'agePct12t29', 'RentHighQ', 'pctWInvInc', 'MalePctNevMarr', 'PctUnemployed', 'PctSameState85', 'PctPersDenseHous', 'racepctblack', 'racePctAsian', 'population', 'pctUrban', 'PersPerOccupHous', 'NumStreet', 'PctSpeakEnglOnly', 'PctRecImmig5', 'PctHousOwnOcc', 'pctWRetire', 'RentMedian', 'PctImmigRec5', 'PctHousLess3BR', 'PctWOFullPlumb', 'agePct16t24', 'AsianPerCap', 'PctForeignBorn', 'PctLess9thGrade', 'medIncome', 'MedRentPctHousInc', 'PctEmploy', 'pctWFarmSelf', 'PersPerOwnOccHous', 'MedOwnCostPctIncNoMtg', 'PctOccupMgmtProf', 'TotalPctDiv', 'PctImmigRec8', 'PctPopUnderPov', 'PctHousOccup', 'RentLowQ', 'PctTeen2Par', 'PctSameCity85', 'NumIlleg', 'PctUsePubTrans', 'pctWWage', 'MedOwnCostPctInc', 'LemasPctOfficDrugUn', 'MedRent', 'PctRecentImmig', 'MalePctDivorce', 'PctVacMore6Mos', 'PctLargHouseFam', 'MedNumBR', 'indianPerCap', 'PctSameHouse85', 'medFamInc', 'numbUrban', '

#### Attributes selection Lasso vs. Forward Attribute Selection:

1) If we take last_best as threshold in forward_attribute_selection we get 11 attributes and they are all included in the lasso selection.
2) If we take average_mse as threshold in forward_attribute_selection we get 78 attributes and they are also all included in the lasso selection.

In [6]:
fas = selected_attributes.columns
lasso_attributes = best_lasso_model.feature_names_in_



model = LinearRegression()
model.fit(X_train[fas], y_train)
fas_predictions = model.predict(X_test[fas])


fas_mae = mean_absolute_error(y_test, fas_predictions)
fas_mse= mean_squared_error(y_test, fas_predictions)

model = LinearRegression()
model.fit(X_train[lasso_attributes], y_train)
lasso_predictions = model.predict(X_test[lasso_attributes])


lasso_mae = mean_absolute_error(y_test, lasso_predictions)
lasso_mse= mean_squared_error(y_test, lasso_predictions)

print("Linear Regression performed with:                   Results:   ")
print(f"Forward Attribute Selection (FAS) -> MAE = {fas_mae}, MSE = {fas_mse}")
print(f"Lasso attributes ->                  MAE = {lasso_mae}, MSE = {lasso_mse}")

Linear Regression performed with:                   Results:   
Forward Attribute Selection (FAS) -> MAE = 0.09504506775825053, MSE = 0.017473761558588978
Lasso attributes ->                  MAE = 0.0954057883974523, MSE = 0.017468095220830565


<h2> Dataset import and preprocessing for wine

In [5]:
df = pd.read_csv("datasets/winequality-white.csv", sep=';', quotechar='"')

df.fillna(df.mean(), inplace=True)


y = df.pop("quality")

X = df

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True)

df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9


<h2> Gradient descend and Stochastic gradient descent

In [6]:
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.01, alpha=1.0, n_iterations=1000):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.n_iterations = n_iterations
        self.theta = None

    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)  # Initialize weights

        for _ in range(self.n_iterations):
            predictions = X.dot(self.theta)
            errors = predictions - y
            
            # Compute the gradient
            gradient = (1 / m) * (X.T.dot(errors)) + self.alpha * self.theta
            
            # Update parameters
            self.theta -= self.learning_rate * gradient

    def predict(self, X):
        return X.dot(self.theta)

In [7]:
class RidgeRegressionSGD:
    def __init__(self, learning_rate=0.01, alpha=1.0, n_iterations=1000):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.n_iterations = n_iterations
        self.theta = None

    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)  # Initialize weights

        for iteration in range(self.n_iterations):
            # Shuffle the data at the start of each epoch
            indices = np.random.permutation(m)
            X_shuffled = X.iloc[indices].reset_index(drop=True)  # Reset index after shuffling
            y_shuffled = y.iloc[indices].reset_index(drop=True)
            
            # Perform SGD by going through each example
            for i in range(m):
                xi = X_shuffled.iloc[i].values.reshape(1, -1)  # Single example as 2D array
                yi = y_shuffled.iloc[i]

                # Prediction for the single example
                prediction = xi.dot(self.theta)
                error = prediction - yi

                # Compute the gradient for this example
                gradient = xi.flatten() * error + self.alpha * self.theta

                # Update the weights
                self.theta -= self.learning_rate * gradient


    def predict(self, X):
        return X.dot(self.theta)

In [10]:
# Ridge regression with Gradient Descent

learning_rates = [0.000001]

best_by_mse =  float('inf')
best_by_mae =  float('inf')
for lr in learning_rates:
    model_rrgd = RidgeRegressionGD(learning_rate=lr, alpha=0.1, n_iterations=1000)

    model_rrgd.fit(X_train, y_train)

    predictions_rrgd = model_rrgd.predict(X_test)

    mse_rrgd = mean_squared_error(y_test, predictions_rrgd)
    mae_rrgd = mean_absolute_error(y_test, predictions_rrgd)

    if(mse_rrgd < best_by_mse):
        best_by_mse = lr
    if(mae_rrgd < best_by_mae):
        best_by_mae = lr
    print(f"Ridge regression with Gradient Descent ->Learning Rate: {lr}        MSE:{mse_rrgd}      MAE:{mae_rrgd}")
print("Best learning rate by metric:")
print(f"Metric\tLR")
print(f"MAE\t{best_by_mae}\nMSE\t{best_by_mse}")




Ridge regression with Gradient Descent ->Learning Rate: 1e-06        MSE:4.125169060384111      MAE:1.6822765909933706
Best learning rate by metric:
Metric	LR
MAE	1e-06
MSE	1e-06


In [ ]:
# Ridge regression with Stochastic Gradient Descent

best_by_mse = float('inf')
best_by_mae = float('inf')
for lr in learning_rates:

    model_rrsgd = RidgeRegressionSGD(learning_rate=0.01, alpha=0.1, n_iterations=1000)
    model_rrsgd.fit(X_train, y_train)

    # Predictions
    predictions_rrsgd = model_rrsgd.predict(X_test)

    mse_rrsgd = mean_squared_error(y_test, predictions_rrsgd)
    mae_rrsgd = mean_absolute_error(y_test, predictions_rrsgd)
    if(mse_rrsgd < best_by_mse):
        best_by_mse = lr
    if(mae_rrsgd < best_by_mae):
        best_by_mae = lr
    print(f"Ridge regression with Stochastic Gradient Descent -> Learning rate:{lr} MSE:{mse_rrsgd}      MAE:{mae_rrsgd}")

print("Best learning rate by metric:")
print(f"Metric\tLR")
print(f"MAE\t{best_by_mae}\nMSE\t{best_by_mse}")

Ridge regression with Stochastic Gradient Descent -> Learning rate:0.1 MSE:0.01942096877867821      MAE:0.09378411312614994
Ridge regression with Stochastic Gradient Descent -> Learning rate:0.01 MSE:0.02577873236124822      MAE:0.13589429696760344
Ridge regression with Stochastic Gradient Descent -> Learning rate:0.001 MSE:0.018281725352786347      MAE:0.09780770271833078
Ridge regression with Stochastic Gradient Descent -> Learning rate:0.0001 MSE:0.019875799658954377      MAE:0.09879597316281406
Best learning rate by metric:
Metric	LR
MAE	0.001
MSE	0.01
